## Config

### Install

In [0]:
#%run /Workspace/Shared/AVANADE_WhatIf/FASE2/LT/00_utility

In [0]:

%run /Workspace/Shared/AVANADE_WhatIf/FASE1/00_utils

In [0]:
%pip install unidecode


In [0]:
%pip install xgboost==3.2.0

In [0]:

from unidecode import unidecode
import unicodedata 

import re
import ast
import numpy as np
import pandas as pd
import mlflow
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from unidecode import unidecode

import warnings

from pandarallel import pandarallel
pandarallel.initialize(nb_workers = 16, progress_bar=False)

### Parameters

In [0]:
MODEL_URI = "runs:/1fee5e0a06194cfaa67b3da265b37a5b/model"

#RAI_TABLE = "ta_coll.whatif.output_palinsesto_rai"
#COMP_TABLE = "ta_coll.whatif.output_palinsesto_competitor"
RAI_TABLE = "ta_coll.whatif.temp_palinsesto_rai"
COMP_TABLE = "ta_coll.whatif.temp_palinsesto_comp"
HIST_TABLE = "ta_coll.whatif.storico_programmi"

OUTPUT_TABLE = "ta_coll.whatif.output_palinsesto_predict"

START_SEC = 21 * 3600 + 30 * 60
END_SEC = 23 * 3600 + 30 * 60

CUTOFF_DAYS = 7
CUTOFF_NUM_OCCURRENCES = 10
MIN_COMPETITOR_OVERLAP = 0.60

### Functions

In [0]:
def _strip_accents(s):
    """à -> a, è -> e (tivu.tv usa diacritici, Auditel li perde)."""
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

# Suffissi editoriali — port da competitor_features._SUFFIXES (in minuscolo)
# + i pattern emersi nel debug del job 11.
_TITLE_SUFFIXES = [
    # Edizioni TG / pagine
    r"\s+edizione\s+straordinaria$", r"\s+ed\s+straordinaria$",
    r"\s+prima\s+pagina$", r"\s+breaking\s+news$",
    r"\s+ultim[ae]?\s+ora(\s+\w+)?$",
    r"\s+ore\s+\d{1,2}(\s+\d{1,2})?$",
    # Stagioni
    r"\s+prima\s+stagione.*$", r"\s+seconda\s+stagione.*$",
    r"\s+terza\s+stagione.*$", r"\s+quarta\s+stagione.*$",
    r"\s+stagione\s+\d+.*$",
    # Cicli stagionali / weekend
    r"\s+il\s+weekend.*$", r"\s+weekend.*$",
    r"\s+estate$", r"\s+cronache\s+d\s+estate$",
    r"\s+sabato$", r"\s+domenica$", r"\s+di\s+piu$",
    # Speciali / varianti
    r"\s+speciale.*$",
    r"\s+1\^?\s*visione$", r"\s+prima\s+visione$",
    r"\s+inizia\s+la\s+sfida.*$", r"\s+le\s+\d+\s+botole.*$",
    r"\s+prima\s+sfida$", r"\s+il\s+torneo\s+dei\s+campioni$",
    r"\s+il\s+torneo$", r"\s+cosa\s+vi\s+siete\s+persi$",
    # Eventi politici / cronaca
    r"\s+elezioni.*$", r"\s+referendum$", r"\s+si\s+no$", r"\s+si\s+o\s+no$",
    r"\s+il\s+bis\s+di\s+trump.*$", r"\s+il\s+ritorno\s+di\s+trump.*$",
    r"\s+la\s+morte\s+del\s+papa$",
    r"\s+diario\s+del\s+giorno.*$", r"\s+diario\s+della.*$",
    # Closing / segment markers
    r"\s+highlights$", r"\s+aftershow$", r"\s+after\s+show$",
    r"\s+buonanotte$", r"\s+saluti$", r"\s+i\s+saluti$",
    r"\s+rewind$", r"\s+compilation.*$", r"\s+tra\s+poco$",
    # Eventi sportivi numerati (Giro d'Italia 109^ edizione 12^ tap)
    r"\s+\d+\s*edizione.*$", r"\s+\d+\s*tap.*$",
    # Anno trailing (Giro d'Italia 2026 -> giro ditalia)
    r"\s+\d{4}$",
]

# Prefissi editoriali — port da get_family_key
_TITLE_PREFIXES = [
    r"^pres\.?\s*",
    r"^anteprima\s+",
    r"^ant\.?\s*",
    r"^i\s+saluti\s+di\s+",
    r"^la\s+buonanotte\s+di\s+",
]

def normalize_title(s):
    """Normalizzazione aggressiva applicata sia a palinsesto tivu.tv che ad hist Auditel.
    Port di `get_family_key` (competitor_features.py) + adattamenti per il debug 11_job_forecast.
    """
    if s is None: return ""
    s = str(s).lower().strip()
    s = _strip_accents(s)

    # 1. Parens: qualsiasi contenuto (lettere, anni, "Diretta", marker Auditel)
    s = re.sub(r"\s*\([^)]*\)\s*", " ", s)

    # 2. Prefissi
    for prefix in _TITLE_PREFIXES:
        s = re.sub(prefix, "", s)

    # 3. Pattern episodi numerati "X - X, N"
    s = re.sub(r"\s+-\s+.+,\s*\d+$", "", s)

    # 4. Abbreviazioni puntate (R.I.S. -> ris) - prima della punteggiatura
    s = re.sub(r"\b([a-z])(\.\s*[a-z])+\.?\b",
               lambda m: m.group(0).replace(".", "").replace(" ", ""), s)

    # 5. Apostrofi -> rimuovi (camera cafe' -> camera cafe, l'eredita -> leredita)
    s = re.sub(r"['\u2019\u2018`]", "", s)

    # 6. Punteggiatura residua -> spazio
    s = re.sub(r"[^a-z0-9 ]", " ", s)

    # 7. Suffissi editoriali (post-punteggiatura per non confondersi con `,`, `-` ecc.)
    for suffix in _TITLE_SUFFIXES:
        s = re.sub(suffix, "", s)

    # 8. Collapse spazi
    return re.sub(r"\s+", " ", s).strip()


MAPPING_MANUALE = {
    # ── Esistenti validi ──
    ("Rai 1", "telegiornale"):              "tg1",
    ("Rai 1", "che tempo fa"):              "meteo 1",
    ("Canale 5", "meteo"):                  "il meteo",
    ("Rete 4", "tg4 telegiornale"):         "tg4",
    ("Italia 1", "macgyver"):               "mac gyver",

    # ── Nuovi (target verificati in hist) ──
    # TG1 con/senza spazio (4371 righe storiche disponibili)
    ("Rai 1", "tg 1"):                      "tg1",
    # RaiNews24: hist usa "rai news" (1660 Rai1, 2308 Rai3)
    ("Rai 1", "rainews24"):                 "rai news",
    ("Rai 3", "rainews24"):                 "rai news",
    # TGR Rai 3 (943-996 righe)
    ("Rai 3", "piazza affari"):             "tgr piazza affari",
    ("Rai 3", "tg regione meteo"):          "tgr meteo",
    # Sky TG24 Buongiorno (137 righe in hist con prefisso "sky")
    ("Tv8", "tg24 buongiorno"):             "sky tg24 buongiorno",
    # Cifra in lettere (solo 2 righe storiche → match flebile ma onesto)
    ("Rete 4", "un esercito di 5 uomini"):  "un esercito di cinque uomini",
}

def apply_manual_mapping(canale, prog_norm):
    return MAPPING_MANUALE.get((canale, prog_norm), prog_norm)

## Loading Palinsesto Storico e Futuro

In [0]:
df_programmi = spark.table(HIST_TABLE).withColumn('ETA_MEDIA', F.col('ETA_MEDIA').cast('float')).toPandas()

df_programmi['Data'] = pd.to_datetime(df_programmi['Data'])
df_programmi['Mese'] = df_programmi['Data'].dt.month
df_programmi['GiornoSettimana'] = df_programmi['Data'].dt.dayofweek
df_programmi['Ora'] = np.round(df_programmi['ORA_INIZIO_TRX'] / 3600, decimals = 0) # sec to hour
df_programmi['Data'] = df_programmi['Data'].dt.date

In [0]:
df_programmi["programma_norm"] = df_programmi["Programma"].apply(normalize_title)
df_programmi["programma_norm"] = [apply_manual_mapping(c, p) for c, p in zip(df_programmi["Canale"], df_programmi["programma_norm"])]
df_programmi = spark.createDataFrame(df_programmi)

In [0]:
df_programmi.display()

In [0]:
rai_future = spark.table(RAI_TABLE)
rai_future = (rai_future.withColumnRenamed('ora', 'Ora')
                        .withColumnRenamed('genere_predominante', 'DES_GENERE_ESTESA_INT')
                        .withColumnRenamed('giorno_settimana', 'GiornoSettimana')
                        .withColumn("ORA_INIZIO_TRX", (F.split("orario_inizio", ":").getItem(0).cast("int") * 3600 +
                                                        F.split("orario_inizio", ":").getItem(1).cast("int") * 60))
                        .withColumn("ORA_FINE_TRX", (F.split("orario_fine", ":").getItem(0).cast("int") * 3600 +
                                                     F.split("orario_fine", ":").getItem(1).cast("int") * 60)))
rai_future = rai_future.drop('fascia_oraria','target_genere','target_eta','share_storico','durata_minuti')

In [0]:
comp_future = spark.table(COMP_TABLE)
comp_future = (comp_future.withColumnRenamed('ora', 'Ora')
                          .withColumnRenamed('genere_predominante', 'DES_GENERE_ESTESA_INT')
                          .withColumnRenamed('giorno_settimana', 'GiornoSettimana')
                          .withColumn("ORA_INIZIO_TRX", (F.split("orario_inizio", ":").getItem(0).cast("int") * 3600 +
                                                F.split("orario_inizio", ":").getItem(1).cast("int") * 60))
                          .withColumn("ORA_FINE_TRX",(F.split("orario_fine", ":").getItem(0).cast("int") * 3600 +
                                              F.split("orario_fine", ":").getItem(1).cast("int") * 60)))
comp_future = comp_future.drop('fascia_oraria','target_genere','target_eta','share_storico','durata_minuti')

In [0]:
df_all = df_programmi.unionByName(rai_future, allowMissingColumns=True)\
                    .unionByName(comp_future, allowMissingColumns=True)
                    
from pyspark.sql.functions import col
df_all = df_all.filter(col('Data') > "2025-05-27")

In [0]:
df_all = df_all.toPandas()

## Creazione Features

In [0]:
print('Computing precise historical stats...')
df_all = df_all.groupby(['programma_norm','Ora','Canale','GiornoSettimana']).parallel_apply(
    compute_stats_last_occurrences, 
    suffix='Precise', 
    cutoff_days=CUTOFF_DAYS, 
    cutoff_num_occurrences=CUTOFF_NUM_OCCURRENCES).reset_index(drop=True)

In [0]:
print('Computing previous and next program stats...')
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    df_all = df_all.groupby(['Canale']).parallel_apply(
        compute_stats_prev_next_program, 
        suffix='Precise').reset_index(drop=True)

df_programmi_pre = spark.createDataFrame(df_all)
df_programmi_pre = df_programmi_pre.where(F.col('StoricoSharePrecise') >= 0.0)

print('Done!')

In [0]:
df_programmi_pre = add_competitor(
    df_programmi_pre, 
    suffix='Precise', 
    min_competitor_overlap=MIN_COMPETITOR_OVERLAP)

for c in df_programmi_pre.columns:
    if c.endswith('Precise'):
        df_programmi_pre = df_programmi_pre.withColumnRenamed(c, c.replace('Precise',''))

In [0]:
df_programmi = df_programmi_pre
df_programmi = df_programmi.withColumn('HighValueShare', F.col('StoricoShare') >= 0.12)
df_programmi = df_programmi.withColumn('Programma_durata', F.col("ORA_FINE_TRX") - F.col("ORA_INIZIO_TRX"))

for c in ['StoricoShareMax','StoricoShareMin','StoricoShareLast','StoricoShare_prev','StoricoShare_next','StoricoShareLast_prev','StoricoShareLast_next']:
    df_programmi = df_programmi.withColumn(c, F.col(c) - F.col('StoricoShare'))

df_programmi = (
    df_programmi
    .withColumn('Ora', F.col('Ora').cast('string'))
    .withColumn('GiornoSettimana', F.col('GiornoSettimana').cast('string'))
    .withColumn('Mese', F.col('Mese').cast('string'))
)

df_programmi = (
    df_programmi.withColumn('StoricoShareUomini_delta', F.col('StoricoShareUomini') * F.col('StoricoShare') - F.col('StoricoShareUomini_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_08_14_delta', F.col('StoricoShare_08_14') * F.col('StoricoShare') - F.col('StoricoShare_08_14_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_15_24_delta', F.col('StoricoShare_15_24') * F.col('StoricoShare') - F.col('StoricoShare_15_24_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_25_64_delta', F.col('StoricoShare_25_64') * F.col('StoricoShare') - F.col('StoricoShare_25_64_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_65_plus_delta', F.col('StoricoShare_65_plus') * F.col('StoricoShare') - F.col('StoricoShare_65_plus_competitor') * F.col('StoricoShare_competitor'))
)



In [0]:
df_programmi = (
    df_programmi.where(F.col('Canale').isin(['Rai 1', 'Rai 2', 'Rai 3']))
    .where(F.col('Ora') >= 21)
    .where(F.col('Data') >= F.current_date())
)


## Check Features

In [0]:
categorical_features = [
    'Canale',
    'DES_GENERE_ESTESA_INT',
    'DES_GENERE_FILM_INT',
    'DES_GENERE_SPORT_INT',
    'DES_MANIFESTAZIONE_SPORT_INT',
    # 'DES_SPECIALITA_SPORT_INT',
    'Canale_competitor',
    'DES_GENERE_ESTESA_INT_competitor',
    'FlgPrimaVisione',
    'FlgPrimaVisioneGen',
    'FlgPrimaVisioneSpec',
    # 'Cod_Genere_Int',
    'Ora',
    'GiornoSettimana',
    'Mese',
]

categorical_features_ohe = {}

for col in categorical_features:
    col_values = df_programmi.select(col).distinct().orderBy(col).collect()
    categorical_features_ohe[col] = []

    for value in col_values:
        if value[0] is None:
            continue
        value_name = value[0].replace(' ', '_').replace('.', '_').replace('-','_')
        df_programmi = df_programmi.withColumn(f'{col}_{value_name}', F.when(F.col(col) == value[0], 1).otherwise(0))

        categorical_features_ohe[col].append(f'{col}_{value_name}')

In [0]:
X_columns = [
    'ORA_INIZIO_TRX',
    'Programma_durata',
    # 'Ora',
    # 'GiornoSettimana',
    # 'Mese',
    # 'StoricoShare',
    'StoricoShareMax',
    'StoricoShareMin',
    'StoricoShareLast',
    # 'StoricoShareStd',
    'StoricoShare_competitor',
    'StoricoShareLast_competitor',
    # 'StoricoShareUomini',
    # 'StoricoShareUomini_competitor',
    'StoricoShareUomini_delta',
    # 'StoricoShare_08_14',
    # 'StoricoShare_08_14_competitor',
    'StoricoShare_08_14_delta',
    # 'StoricoShare_15_24',
    # 'StoricoShare_15_24_competitor',
    'StoricoShare_15_24_delta',
    # 'StoricoShare_25_64',
    # 'StoricoShare_25_64_competitor',
    'StoricoShare_25_64_delta',
    # 'StoricoShare_65_plus',
    # 'StoricoShare_65_plus_competitor',
    'StoricoShare_65_plus_delta',
    # 'overlap_perc_competitor',
    'StoricoShare_prev',
    'StoricoShare_next',
    'StoricoShareLast_prev',
    'StoricoShareLast_next',
    'StoricoShareUomini_prev',
    'StoricoShareUomini_next',
    'HighValueShare',
    # 'LowValueShare',
    # 'StoricoEtaMedia',
    # 'Share',
]

for cat_ohe, values in categorical_features_ohe.items():
    X_columns.extend(values)

In [0]:
import json
with open('/Workspace/Shared/AVANADE_WhatIf/FASE1/X_columns.json', 'r') as f:
    X_columns = json.load(f)

with open('/Workspace/Shared/AVANADE_WhatIf/FASE1/categorical_features_ohe.json', 'r') as f:
    categorical_features_ohe = json.load(f)

## Load Model

In [0]:
# ==========================================
# LOAD MODEL
# ==========================================

model = mlflow.pyfunc.load_model(MODEL_URI)

In [0]:
# ==========================================
# LOAD TRAINING OHE MAPPING
# ==========================================

with open('/Workspace/Shared/AVANADE_WhatIf/FASE1/categorical_features_ohe.json', 'r') as f:
    categorical_features_ohe = json.load(f)

# ==========================================
# CREATE OHE FEATURES
# ==========================================

for col, mapping in categorical_features_ohe.items():

    print(f'OHE inference: {col}')

    for original_value, feature_name in mapping.items():

        df_programmi = df_programmi.withColumn(
            feature_name,
            F.when(
                F.col(col).cast('string') == F.lit(original_value),
                1
            ).otherwise(0)
        )

# ==========================================
# CONVERT TO PANDAS
# ==========================================

inference_pd = df_programmi.toPandas()

# ==========================================
# READ MODEL SIGNATURE
# ==========================================

signature = model.metadata.signature

expected_columns = [
    col.name
    for col in signature.inputs.inputs
]

print(f'Expected cols: {len(expected_columns)}')

# ==========================================
# ALIGN FEATURE MATRIX
# ==========================================

X_inf = inference_pd.reindex(
    columns=expected_columns,
    fill_value=0
)

## ==========================================
# FORCE TYPES FROM MLFLOW SIGNATURE
# ==========================================

type_mapping = {
    "integer": "int32",
    "long": "int64",
    "float": "float32",
    "double": "float64",
    "boolean": "bool"
}

signature_types = {
    col.name: col.type.name
    for col in signature.inputs.inputs
}

for col_name, mlflow_type in signature_types.items():

    pandas_type = type_mapping.get(mlflow_type, "float64")

    X_inf[col_name] = pd.to_numeric(
        X_inf[col_name],
        errors='coerce'
    )

    if "int" in pandas_type:

        X_inf[col_name] = (
            X_inf[col_name]
            .fillna(0)
            .astype(pandas_type)
        )

    else:

        X_inf[col_name] = (
            X_inf[col_name]
            .astype(pandas_type)
        )

# final cleanup
X_inf = X_inf.fillna(0)





In [0]:
# ==========================================
# DEBUG OPTIONAL
# ==========================================

extra_cols = set(inference_pd.columns) - set(expected_columns)
missing_cols = set(expected_columns) - set(inference_pd.columns)

print(f'Extra cols ignored: {len(extra_cols)}')
print(f'Missing cols added: {len(missing_cols)}')


In [0]:
extra_cols

In [0]:
missing_cols

# Predict

In [0]:
# ==========================================
# PREDICT
# ==========================================

pred_residuo = model.predict(X_inf)

inference_pd['share_residuo'] = pred_residuo

inference_pd['share_predetto'] = (
    inference_pd['StoricoShare'] +
    inference_pd['share_residuo']
)

# Output

In [0]:

# ==========================================
# OUTPUT
# ==========================================
from pyspark.sql import functions as F

output = spark.createDataFrame(inference_pd)

out = output.select(
    'Canale',
    'Data',
    'Programma',
    'programma_norm',
    'ORA_INIZIO_TRX',
    'ORA_FINE_TRX',
    'orario_inizio',
    'orario_fine',
    'StoricoShare',
    'share_residuo',
    'share_predetto',
    'Canale_competitor',
    'Programma_competitor',
    'Programma_prev',
    'Programma_next')

display(out)

In [0]:
# out.write.mode("overwrite").option("mergeSchema", "true").saveAsTable('ta_coll.whatif.output_palinsesto_predict')

df_new_out = spark.createDataFrame(out)
# delta_table_out = spark.table('ta_coll.whatif.output_palinsesto_predict')

# (
#     delta_table_out.alias("target")
#     .merge(
#         df_new_out.alias("source"),
#         "target.Data = source.Data AND target.Canale = source.Canale AND target.orario_inizio = source.orario_inizio AND target.programma_norm = source.programma_norm"
#     )
#     .whenMatchedUpdateAll()
#     .whenNotMatchedInsertAll()
#     .execute()
# )

In [0]:
%sql
select *
from ta_coll.whatif.out_palinsesto_predict

In [0]:
%sql
select *
from ta_coll.whatif.out_palinsesto_predict


In [0]:
from pyspark.sql.types import DoubleType
import pyspark.sql.functions as f

out_palinsesto_predict = spark.table('ta_coll.whatif.out_palinsesto_predict')
out_palinsesto_predict = out_palinsesto_predict.withColumn('share_manuale', f.lit(None).cast(DoubleType()))
out_palinsesto_predict.display()

In [0]:
out_palinsesto_predict.write.mode("overwrite").option("mergeSchema", "true").saveAsTable('ta_coll.whatif.out_palinsesto_predict')